# Edge IIoT - Data Cleaning for Machine Learning

This phase focuses on refining the raw dataset into a structured format suitable for subsequent modeling. The following principles guide this process:
- The cleaning procedures described are applicable to both Binary and Multiclass classification tasks. Note that specific features may be selectively enabled or disabled depending on the classification objective and restrictions.
- In this study, "Cleaning" includes basic feature transformations that can be performed during the initial packet capture (PCAP) extraction or via simple logic. Complex, model-dependent transformations (such as scaling, normalization, or advanced encoding) are reserved for the Preprocessing phase.
- The cleaning logic is informed by the following kaggle notebooks:
    - [Edge-IIoTset Pre-Processing](https://www.kaggle.com/code/mohamedamineferrag/edge-iiotset-pre-processing) by Mohamed Amine Ferrag
    - [Predict Attack and Attack Type](https://www.kaggle.com/code/waleedgul/predict-attack-and-attack-type#Irrelevant-Columns-for-IoT-Attack-Detection) by Waleed Gul

In [1891]:
LOAD_LARGE_CSV = True # Set to True if you want to load the DNN file instead of the ML one.

## 0 - Obtaining data and initial exploration

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.config import DATASETS

# Seleccionamos el dataset a analizar
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['raw_path']}\n")

if LOAD_LARGE_CSV:
    filename = "DNN-EdgeIIoT-dataset"
else:
    filename = "ML-EdgeIIoT-dataset"

csv_path = config['raw_path'] / "Edge-IIoTset dataset" / "Selected dataset for ML and DL" / f"{filename}.csv"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_csv(csv_path, low_memory=False)

print(f"\nDataset loaded with shape: {df.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/raw/edge_iiot

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/raw/edge_iiot/Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv

Loading dataset... (This may take a while)

Dataset loaded with shape: (157800, 63)


In [1893]:
df.loc[0, (df != '0').any(axis=0)]

frame.time                      6.0
ip.src_host           192.168.0.152
ip.dst_host                     0.0
arp.dst.proto_ipv4              0.0
arp.opcode                      0.0
                          ...      
mbtcp.len                       0.0
mbtcp.trans_id                  0.0
mbtcp.unit_id                   0.0
Attack_label                      1
Attack_type                    MITM
Name: 0, Length: 63, dtype: object

In [1894]:
print("Column count:", len(df.columns))
print("\n--- Columns ---\n", df.columns)

Column count: 63

--- Columns ---
 Index(['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4',
       'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum',
       'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused',
       'http.file_data', 'http.content_length', 'http.request.uri.query',
       'http.request.method', 'http.referer', 'http.request.full_uri',
       'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack',
       'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin',
       'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack',
       'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options',
       'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream',
       'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu',
       'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request',
       'dns.retransmit_request_in', 'mqtt.conack.flags',
       'mqtt.conflag.cleansess', 'mqtt.co

In [1895]:
packet_types = []
for col in df.columns:
    if '.' in col:
        proto = col.split('.')[0]
        if proto not in packet_types:
            packet_types.append(proto)

print(f"\nPacket types: {packet_types}\n")


Packet types: ['frame', 'ip', 'arp', 'icmp', 'http', 'tcp', 'udp', 'dns', 'mqtt', 'mbtcp']



## 1 - Treating Duplicates

In [1896]:
print("Num. duplicates:", df.duplicated().sum())

Num. duplicates: 814


In [1897]:
df.drop_duplicates(subset=None, keep="first", inplace=True)
print("Num. duplicates after dropping:", df.duplicated().sum())

Num. duplicates after dropping: 0


## 2 - Treating Columns

In [1898]:
def print_unique_values(df, column_names):
    for col in column_names:
        unique_values = df[col].unique()
        print(f"\n--- {col} --- \n{unique_values}\n")

def find_packet_type_columns(df, packet_type):
    return [col for col in df.columns if col.startswith(f'{packet_type}.')]

### 2.01 - Frame

In [1899]:
cols = find_packet_type_columns(df, 'frame')
print("\n>>> FRAME COLUMNS <<<\n", cols)


>>> FRAME COLUMNS <<<
 ['frame.time']


#### 2.01.01 - `frame.time`
- **Definition:** Logs the precise arrival timestamp of the network packet at the Frame layer.
- **Treatment:** Dropped.
- **Reason:** Timing alone doesn’t indicate malicious behavior. ML models will model only spacial features.

In [1900]:
df.drop(columns=['frame.time'], inplace=True)
print("\nDropped 'frame.time' column:", not 'frame.time' in df.columns)


Dropped 'frame.time' column: True


### 2.02 - IP (Internet Protocol)

In [1901]:
cols = find_packet_type_columns(df, 'ip')
print("\n>>> IP COLUMNS <<<\n", cols)


>>> IP COLUMNS <<<
 ['ip.src_host', 'ip.dst_host']


#### 2.02.01 - `ip.src_host` and `ip.dst_host`

- **Definition**: Source and destination IPv4 addresses of the packet.
- **Treatment**: Dropped.
- **Reason**: High risk of spoofing and poor generalizability. Removal prevents the model from memorizing environment-specific addresses.

In [1902]:
df.drop(columns=['ip.src_host', 'ip.dst_host'], inplace=True)
print("\nDropped 'ip.src_host' column:", not 'ip.src_host' in df.columns)
print("\nDropped 'ip.dst_host' column:", not 'ip.dst_host' in df.columns)


Dropped 'ip.src_host' column: True

Dropped 'ip.dst_host' column: True


### 2.03 - ARP (Address Resolution Protocol)

In [1903]:
cols = find_packet_type_columns(df, 'arp')
print("\n>>> ARP COLUMNS <<<\n", cols)


>>> ARP COLUMNS <<<
 ['arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4']


#### 2.03.01 - `arp.src.proto_ipv4` and `arp.dst.proto_ipv4`

- **Definition**: Represents the sender and target IPv4 addresses within the ARP header.
- **Treatment**: Dropped.
- **Reason**: Removal prevents the model from memorizing environment-specific addresses.

In [1904]:
df.drop(columns=['arp.dst.proto_ipv4', 'arp.src.proto_ipv4'], inplace=True)
print("\nDropped 'arp.dst.proto_ipv4' column:", not 'arp.dst.proto_ipv4' in df.columns)
print("\nDropped 'arp.src.proto_ipv4' column:", not 'arp.src.proto_ipv4' in df.columns)


Dropped 'arp.dst.proto_ipv4' column: True

Dropped 'arp.src.proto_ipv4' column: True


#### 2.03.02 - `arp.hw.size`

- **Definition**: Specifies the length (in bytes) of the hardware address.
- **Treatment**: Dropped.
- **Reason**: Redundancy and zero variance. The value remains constant (`6`) for all valid ARP packets, resulting in high collinearity with arp.opcode.

In [1905]:
print_unique_values(df, ['arp.hw.size'])


--- arp.hw.size --- 
[0. 6.]



In [1906]:
print(
    "\n--- Count hw.size if arp.code != 0 ---\n",
    df[df['arp.opcode'] != 0.]['arp.hw.size'].value_counts(),
    "\n\n\n--- Correlation opcode to hw.size ---\n",
    f"{df[['arp.opcode', 'arp.hw.size']].corr()['arp.hw.size']['arp.opcode']:0.4f}"
)


--- Count hw.size if arp.code != 0 ---
 arp.hw.size
6.0    1574
Name: count, dtype: int64 


--- Correlation opcode to hw.size ---
 0.9442


In [1907]:
df.drop(columns=['arp.hw.size'], inplace=True)
print("\nDropped 'arp.hw.size' column:", not 'arp.hw.size' in df.columns)


Dropped 'arp.hw.size' column: True


#### 2.03.03 - `arp.opcode`

- **Definition**: Type of ARP message (`0` = No ARP, `1` = ARP Request or `2` = ARP Reply).
- **Treatment**: Retained / Binarization. Transformed into two binary features: `arp.opcode_request` and `arp.opcode_reply`.
- **Reason**: The opcode is a nominal categorical variable, not an ordinal one.

> NOTE
> 
> Non-ARP packets (original value 0) result in both new binary features being set to 0.

In [1908]:
print_unique_values(df, ['arp.opcode'])


--- arp.opcode --- 
[0. 1. 2.]



In [1909]:
df['arp.opcode_request'] = (df['arp.opcode'] == 1).astype(int)
df['arp.opcode_reply'] = (df['arp.opcode'] == 2).astype(int)

print(df[['arp.opcode_request', 'arp.opcode_reply']].sum())

arp.opcode_request    908
arp.opcode_reply      666
dtype: int64


In [1910]:
df.drop(columns=['arp.opcode'], inplace=True)
print("\nDropped 'arp.opcode' column:", not 'arp.opcode' in df.columns)


Dropped 'arp.opcode' column: True


### 2.04 - ICMP (Internet Control Message Protocol)

In [1911]:
cols = find_packet_type_columns(df, 'icmp')
print("\n>>> ICMP COLUMNS <<<\n", cols)


>>> ICMP COLUMNS <<<
 ['icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused']


#### 2.04.01 - `icmp.checksum`

- **Definition**: A 16-bit field used for error-checking the ICMP header and data to ensure integrity.
- **Treatment**: Dropped.
- **Reason:** Checksums are transient values computed dynamically per packet.

In [1912]:
print_unique_values(df, ['icmp.checksum'])


--- icmp.checksum --- 
[    0. 11938. 13986. ... 45657. 57686.  9555.]



In [1913]:
df.drop(columns=['icmp.checksum'], inplace=True)
print("\nDropped 'icmp.checksum' column:", not 'icmp.checksum' in df.columns)


Dropped 'icmp.checksum' column: True


#### 2.04.02 - `icmp.seq_le`

- **Definition**: The Sequence Number (Little Endian) used to pair Echo Requests with corresponding Echo Replies.
- **Treatment**: Retained
- **Reason:** Discontinuities, high-frequency increments, or non-sequential jumps in sequence numbers are primary indicators of ICMP flooding (DoS) or data exfiltration via covert tunneling.

In [1914]:
print_unique_values(df, ['icmp.seq_le'])


--- icmp.seq_le --- 
[    0.   256.  8154. ... 40702. 41423. 42379.]



#### 2.04.03 - `icmp.transmit_timestamp`

- **Definition**: A timestamp indicating when the sender last interacted with the message before transmission.
- **Treatment**: Dropped.
- **Reason:** Timing synchronization data is highly host-specific. ML models will model only spacial features.

In [1915]:
df.drop(columns=['icmp.transmit_timestamp'], inplace=True)
print("\nDropped 'icmp.transmit_timestamp' column:", not 'icmp.transmit_timestamp' in df.columns)


Dropped 'icmp.transmit_timestamp' column: True


#### 2.04.04 - `icmp.unused`

- **Definition**: A 4-byte reserved field that is mandated to be set to zero in specific ICMP message types.
- **Treatment**: Dropped.
- **Reason:** By definition, this field contains no information. This column is constant to `0.`.

In [1916]:
print_unique_values(df, ['icmp.unused'])


--- icmp.unused --- 
[0.]



In [1917]:
df.drop(columns=['icmp.unused'], inplace=True)
print("\nDropped 'icmp.unused' column:", not 'icmp.unused' in df.columns)


Dropped 'icmp.unused' column: True


### 2.05 - HTTP (HyperText Transfer Protocol)

In [1918]:
cols = find_packet_type_columns(df, 'http')
print("\n>>> HTTP COLUMNS <<<\n", cols)


>>> HTTP COLUMNS <<<
 ['http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port']


#### 2.05.01 - `http.file_data`

- **Definition**: The actual body content or payload transmitted within an HTTP entity.
- **Treatment**: Retained / Replaced `'0.0'` and `'0'` to `''`
- **Reason:** Although computionally expensive, could be an essential feature for multiclass clasification. We can try extracting the observed size to detect buffer overflow attacks.

In [1919]:
print_unique_values(df, ['http.file_data'])


--- http.file_data --- 
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [1920]:
df['http.file_data'] = df['http.file_data'].replace(['0', '0.0'], '')

#### 2.05.02 - `http.content_length`

- **Definition**: The size of the entity-body, in decimal number of octets, sent to the recipient.
- **Treatment**: Retained
- **Reason:** Large discrepancies between content_length and actual observed bytes can indicate data exfiltration or buffer overflow attempts.

> NOTE
>
> High Standard Desviation is observed.
>

In [1921]:
print_unique_values(df, ['http.content_length'])


--- http.content_length --- 
[0.0000e+00 2.9800e+02 3.0300e+02 1.4650e+03 1.4150e+03 3.7000e+01
 3.8000e+01 2.7700e+02 2.7300e+02 3.1500e+02 3.6000e+01 2.8000e+02
 1.0000e+00 1.2000e+01 5.7000e+01 1.1500e+02 5.0000e+00 7.1400e+02
 4.4000e+01 2.2000e+01 3.9000e+01 8.6000e+01 6.0000e+00 1.6400e+02
 4.7000e+01 3.0700e+02 1.4040e+03 3.0100e+02 1.1550e+03 5.9000e+01
 8.3655e+04 2.6200e+02 2.2900e+02]



In [1922]:
df['http.content_length'].describe()

count    156986.000000
mean         14.791822
std         230.251867
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       83655.000000
Name: http.content_length, dtype: float64

#### 2.05.03 - `http.request.uri.query`

- **Definition**: The query string portion of the URI (e.g., everything after the ?).
- **Treatment**: Retained / Replaced `'0.0'` and `'0'` to `''`
- **Reason:** Essential to detect SQL injection or XSS patterns. The raw string is too high-cardinality for direct use.

In [1923]:
print_unique_values(df, ['http.request.uri.query'])


--- http.request.uri.query --- 
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                             '0.0',
                                                                                                                                                                                                                                                                                                                                                                '0',
                                                                                                                                                                                                                                               

In [1924]:
df['http.request.uri.query'] = df['http.request.uri.query'].replace(['0', '0.0'], '')

#### 2.05.04 - `http.request.method`

- **Definition**: The HTTP verb used (GET, POST, PUT, DELETE, etc.).
- **Treatment**: Binarization. Transformed into the binary features: `method_get`, `method_head`, `method_post`, `method_put`, `method_delete`, `method_connect`, `method_options`, `method_trace` and `method_patch`.
- **Reason:** Methods are nominal categories. Identifying "POST" vs "GET" is essential, as certain attacks (like credential stuffing) are almost exclusively performed via POST requests.

> NOTE
>
> Although some methods are not present in the dataset, they are included for latter real-world testing.

In [1925]:
print_unique_values(df, ['http.request.method'])


--- http.request.method --- 
<StringArray>
['0.0', '0', 'GET', 'POST', 'OPTIONS', 'TRACE']
Length: 6, dtype: str



In [1926]:
methods = {
    'GET': 'http.request.method_get',
    'HEAD': 'http.request.method_head',
    'POST': 'http.request.method_post',
    'PUT': 'http.request.method_put',
    'DELETE': 'http.request.method_delete',
    'CONNECT': 'http.request.method_connect',
    'OPTIONS': 'http.request.method_options',
    'TRACE': 'http.request.method_trace',
    'PATCH': 'http.request.method_patch'
}

active_http_columns = []
for method, col_name in methods.items():
    if (df['http.request.method'] == method).any():
        df[col_name] = (df['http.request.method'] == method).astype(int)
        active_http_columns.append(col_name)

print(df[active_http_columns].sum())

http.request.method_get        6676
http.request.method_post        267
http.request.method_options       1
http.request.method_trace       252
dtype: int64


In [1927]:
df.drop(columns=['http.request.method'], inplace=True)
print("\nDropped 'http.request.method' column:", not 'http.request.method' in df.columns)


Dropped 'http.request.method' column: True


#### 2.05.05 - `http.referer`

- **Definition**: Identifies the address of the webpage that linked to the requested resource.
- **Treatment**: Retained / Replaced `'0.0'` and `'0'` to `''`
- **Reason:** Critical indicator of a Remote Code Execution attack (such as `() { _; } >_[$($())] { echo 93e4r0... }`).

In [1928]:
print_unique_values(df, ['http.referer'])


--- http.referer --- 
<StringArray>
[                                                                 '0.0',
                                                                    '0',
 '() { _; } >_[$($())] { echo 93e4r0-CVE-2014-6278: true; echo;echo; }',
                                                            '127.0.0.1']
Length: 4, dtype: str



In [1929]:
df['http.referer'] = df['http.referer'].replace(['0', '0.0'], '')

#### 2.05.06 - `http.request.full_uri`

- **Definition**: The complete URI string, encompassing the scheme, host/domain, and path.
- **Treatment**: Dropped / Feature Extraction (Path Only).
- **Reason:** The query component is already captured in `http.request.uri.query`, and hostnames vary inconsistently across different environments. Only the path component is retained or extracted, as it typically contains the target endpoint and potential attack vectors.

In [1930]:
print_unique_values(df, ['http.request.full_uri'])


--- http.request.full_uri --- 
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                   '0.0',
                                                                                                                                                                                                                                                                                                                                                                                                                      '0',
                                                                                                                                    

In [1931]:
from urllib.parse import urlparse

def extract_path(full_uri):
    try:
        return urlparse(full_uri).path
    except Exception:
        return ''

# Apply to your dataframe
df['http.request.path'] = df['http.request.full_uri'].apply(extract_path)

In [1932]:
print_unique_values(df, ['http.request.path'])


--- http.request.path --- 
<StringArray>
[                                     '0.0',
                                        '0',
          '/DVWA/hackable/uploads/hack.php',
        '/DVWA/vulnerabilities/sqli_blind/',
                          '/DVWA/login.php',
                       '/DVWA/fd5ljfpL.ASP',
                      '/DVWA/fd5ljfpL.json',
                        '/DVWA/fd5ljfpL.tw',
                       '/DVWA/fd5ljfpL.xbb',
                      '/DVWA/fd5ljfpL.php=',
 ...
     '/DVWA/plugin/gateway/gnokii/init.php',
                      '/DVWA/poll/view.php',
                      '/DVWA/principal.php',
                     '/DVWA/redsys/404.php',
 '/DVWA/script_path/installation/index.php',
                  '/DVWA/show_archives.php',
     '/DVWA/source/mod/rss/channeledit.php',
 '/DVWA/speedberg/include/scriplet.inc.php',
           '/DVWA/supasite/admin_users.php',
                   '/dvwa/vulnerabilities/']
Length: 3024, dtype: str



In [1933]:
df['http.request.path'] = df['http.request.path'].replace(['0', '0.0'], '')

In [1934]:
df.drop(columns=['http.request.full_uri'], inplace=True)
print("\nDropped 'http.request.full_uri' column:", not 'http.request.full_uri' in df.columns)


Dropped 'http.request.full_uri' column: True


#### 2.05.07 - `http.request.version`

- **Definition**: The version of the HTTP protocol used (e.g., HTTP/1.1, HTTP/2).
- **Treatment**: Retained
- **Reason:** Anomalous or outdated versions can be an indicator of an attack.

In [1935]:
print_unique_values(df, ['http.request.version'])


--- http.request.version --- 
<StringArray>
[                                                      '0',
                                                     '0.0',
                                                'HTTP/1.1',
                                                'HTTP/1.0',
 'Src=javascript:alert('Vulnerable')><Img Src=\" HTTP/1.1',
                     '/etc/passwd|?data=Download HTTP/1.1',
                                             '-a HTTP/1.1',
                                          'By Dr HTTP/1.1']
Length: 8, dtype: str



In [1936]:
df['http.request.version'] = df['http.request.version'].replace(['0', '0.0'], '')

#### 2.05.08 - `http.response`

- **Definition**: A binary flag (`0` or `1`) indicating whether the packet is a response.
- **Treatment**: Retained.
- **Reason:** Asymmetrical request-response flows can be an indicator of DoS attacks.

In [1937]:
print_unique_values(df, ['http.response'])


--- http.response --- 
[0. 1.]



#### 2.05.09 - `http.tls_port`

- **Definition**: The port number used if the HTTP traffic is encapsulated in TLS (HTTPS).
- **Treatment**: Retained
- **Reason:** Non-standard TLS ports are frequently used by malware for Command and Control (C2) communication to bypass basic firewall filters.

> NOTE
>
> In this case the value is constant to `0` as there are no encrypted HTTPS traffic. Retained for structural integrity.

In [1938]:
print_unique_values(df, ['http.tls_port'])


--- http.tls_port --- 
[0.]



### 2.06 - TCP (Transmission Control Protocol)

In [1939]:
cols = find_packet_type_columns(df, 'tcp')
print("\n>>> TCP COLUMNS <<<\n", cols)


>>> TCP COLUMNS <<<
 ['tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport']


#### 2.06.01 - `tcp.ack` (Relative)

- **Definition**: A 32-bit field that indicates the next sequence number the sender of the segment expects to receive. 
- **Treatment**: Retained
- **Reason:** Helps detecting anomalies in communication flows.

> NOTE
>
> "Relative," means that it starts at 1 for each new connection to simplify tracking.

In [1940]:
print_unique_values(df, ['tcp.ack'])


--- tcp.ack --- 
[0.000000e+00 1.000000e+00 4.650000e+02 ... 5.994378e+06 2.900000e+01
 1.400000e+01]



#### 2.06.02 - `tcp.ack_raw`

- **Definition**: The actual 32-bit acknowledgment number contained in the TCP header, representing the absolute value of the next byte expected. 
- **Treatment**: Dropped
- **Reason:** It provides redundant information when tcp.ack (relative) is already present.

In [1941]:
df.drop(columns=['tcp.ack_raw'], inplace=True)
print("\nDropped 'tcp.ack_raw' column:", not 'tcp.ack_raw' in df.columns)


Dropped 'tcp.ack_raw' column: True


#### 2.06.03 - `tcp.checksum`

- **Definition**: A 16-bit field in the TCP header used for error-checking to ensure the integrity of the segment's header and data during transmission.
- **Treatment**: Dropped
- **Reason:** Transient value that varies with every change in the payload or header.

In [1942]:
df.drop(columns=['tcp.checksum'], inplace=True)
print("\nDropped 'tcp.checksum' column:", not 'tcp.checksum' in df.columns)


Dropped 'tcp.checksum' column: True


#### 2.06.04 - `tcp.dstport` and `tcp.srcport`

- **Definition**: The source and destination ports.
- **Treatment**: Dropped
- **Reason:** Prevents the model from overfitting to specific network configurations or service assignments.

In [1943]:
for col in ['tcp.dstport', 'tcp.srcport']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'tcp.dstport' column: True

Dropped 'tcp.srcport' column: True


#### 2.06.05 - `tcp.flags`

- **Definition**: TCP control bits.
- **Treatment**: Binarized.
- **Reason:** Critial for detecting DoS/DDoS attacks, such as TCP SYN Floods.

In [1944]:
tcp_flags_map = {
        'tcp.flag.res': 0x200, # Reserved
        'tcp.flag.ns':  0x100, # NS
        'tcp.flag.cwr': 0x080, # CWR
        'tcp.flag.ece': 0x040, # ECE
        'tcp.flag.urg': 0x020, # Urg
        'tcp.flag.ack': 0x010, # Ack
        'tcp.flag.psh': 0x008, # Push
        'tcp.flag.rst': 0x004, # Reset
        'tcp.flag.syn': 0x002, # Syn
        'tcp.flag.fin': 0x001  # Fin
    }

df['tcp.flags'] = df['tcp.flags'].astype(int)

for flag_name, bit_mask in tcp_flags_map.items():
    df[flag_name] = (df['tcp.flags'] & bit_mask).gt(0).astype('uint8')

In [1945]:
df.drop(columns=['tcp.flags'], inplace=True)
print("\nDropped 'tcp.flags' column:", not 'tcp.flags' in df.columns)


Dropped 'tcp.flags' column: True


#### 2.06.06 - `tcp.connection.fin`, `tcp.connection.rst`, `tcp.connection.syn`, `tcp.connection.synack`, `tcp.flags.ack`

- **Definition**: TCP control bits.
- **Treatment**: Dropped.
- **Reason:** Redundant.

> NOTE
>
>  The column `tcp.connection.syn` does not perfectly correlate to new `tcp.flag.syn` because the first is not the bit control itself but the condition `SYN = 1 AND ACK = 0`.

In [1946]:
for col in ['tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.flags.ack']:
    print(f"{col:>25}: {df[df['tcp.ack'] > 0][col].unique()}")

       tcp.connection.fin: [0. 1.]
       tcp.connection.rst: [1. 0.]
       tcp.connection.syn: [0. 1.]
    tcp.connection.synack: [0. 1.]
            tcp.flags.ack: [1. 0.]


In [1947]:
print("--- Correlation between TCP flags ---")
print(f"{"FIN":>10}: {df[['tcp.connection.fin', 'tcp.flag.fin']].corr()['tcp.flag.fin']['tcp.connection.fin']:0.4f}")
print(f"{"RST":>10}: {df[['tcp.connection.rst', 'tcp.flag.rst']].corr()['tcp.flag.rst']['tcp.connection.rst']:0.4f}")
print(f"{"SYN":>10}: {df[['tcp.connection.syn', 'tcp.flag.syn']].corr()['tcp.flag.syn']['tcp.connection.syn']:0.4f}")
print(f"{"ACK":>10}: {df[['tcp.flags.ack',      'tcp.flag.ack']].corr()['tcp.flag.ack']['tcp.flags.ack'     ]:0.4f}")

--- Correlation between TCP flags ---
       FIN: 1.0000
       RST: 1.0000
       SYN: 0.8844
       ACK: 1.0000


In [1948]:
for col in ['tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.flags.ack']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'tcp.connection.fin' column: True

Dropped 'tcp.connection.rst' column: True

Dropped 'tcp.connection.syn' column: True

Dropped 'tcp.connection.synack' column: True

Dropped 'tcp.flags.ack' column: True


#### 2.06.07 - `tcp.len`

- **Definition**: A field that indicates the size of the TCP segment data (payload) in bytes, excluding the TCP header.
- **Treatment**: Retained.
- **Reason:** Used to detect volumetric attacks. Sudden changes in packet size or a high frequency of identical lengths are indicators of DoS/DDoS flooding.

In [1949]:
df[df['tcp.len'] > 0]['tcp.len'].describe()

count    42306.000000
mean       484.072377
std       2490.116964
min          2.000000
25%        120.000000
50%        288.000000
75%        464.000000
max      65228.000000
Name: tcp.len, dtype: float64

#### 2.06.08 - `tcp.options`

- **Definition**: Optional parameters in the TCP header.
- **Treatment**: Dropped.
- **Reason:** These are low-level TCP configuration details that generally add noise rather than useful signals for detection.

In [1950]:
print_unique_values(df, ['tcp.options'])


--- tcp.options --- 
<StringArray>
[                     '0.0',                     '67.0',
                   '5355.0',                   '5353.0',
                    '123.0',                  '57342.0',
                    '137.0',                  '37691.0',
 '0101080ade2d27b4066ccff1', '0101080a066cd98bde2d27b4',
 ...
 '0101050a01a92ba001a92ba1', '0101050acbd49642cbd49643',
 '0101050aa10a8e80a10a8e81', '0101050a9213204492132045',
 '0101050a887a3ef6887a3ef7', '0101050a2b9741352b974136',
 '0101050a516f4d36516f4d37', '0101050a6decedb16decedb2',
 '0101050a0638936206389363', '0101050a76b39cac76b39cb0']
Length: 73139, dtype: str



In [1951]:
df.drop(columns=['tcp.options'], inplace=True)
print("\nDropped 'tcp.options' column:", not 'tcp.options' in df.columns)


Dropped 'tcp.options' column: True


#### 2.06.09 - `tcp.payload`

- **Definition**: The actual data content carried by the TCP segment.
- **Treatment**: Dropped. 
- **Reason:** The authors of the dataset explicitly recommend dropping payload information to ensure privacy and focus on statistical network behavior.

In [1952]:
df.drop(columns=['tcp.payload'], inplace=True)
print("\nDropped 'tcp.payload' column:", not 'tcp.payload' in df.columns)


Dropped 'tcp.payload' column: True


#### 2.06.10 - `tcp.seq`

- **Definition**: A 32-bit numerical value assigned to the first byte of data in a segment.
- **Treatment**: Retained.
- **Reason:** Manipulations or high values are indicators of sequence-based and flooding attacks.

In [1953]:
print_unique_values(df, ['tcp.seq'])


--- tcp.seq --- 
[0.0000e+00 1.5011e-02 1.7182e-02 ... 3.5874e+04 1.6000e+01 2.8000e+01]



In [1954]:
df[df['tcp.seq'] > 0]['tcp.seq'].describe()

count    9.929200e+04
mean     2.980024e+06
std      1.983185e+07
min      4.000000e-05
25%      1.000000e+00
50%      1.500000e+01
75%      8.140000e+02
max      2.079647e+08
Name: tcp.seq, dtype: float64

### 2.07 - UDP (User Datagram Protocol)

In [1955]:
cols = find_packet_type_columns(df, 'udp')
print("\n>>> UDP COLUMNS <<<\n", cols)


>>> UDP COLUMNS <<<
 ['udp.port', 'udp.stream', 'udp.time_delta']


#### 2.07.01 - `udp.port`

- **Definition**: A numerical identifier (16-bit) used to distinguish between different services or processes communicating via the User Datagram Protocol.
- **Treatment**: Dropped.
- **Reason:** Similar to TCP ports. Removed to prevent model from overfitting.

In [1956]:
df.drop(columns=['udp.port'], inplace=True)
print("\nDropped 'udp.port' column:", not 'udp.port' in df.columns)


Dropped 'udp.port' column: True


#### 2.07.02 - `udp.stream`

- **Definition**: An index that groups related UDP packets belonging to the same flow between two endpoints.
- **Treatment**: Retained.
- **Reason:** Allows to track the consistency of a data flow.

In [1957]:
print_unique_values(df, ['udp.stream'])


--- udp.stream --- 
[0.000000e+00 1.230000e+02 5.300000e+01 ... 2.897710e+06 2.897835e+06
 2.898725e+06]



In [1958]:
df[df['udp.stream'] > 0]['udp.stream'].describe()

count    1.453600e+04
mean     1.315078e+06
std      9.029611e+05
min      1.200000e+01
25%      4.968718e+05
50%      1.302127e+06
75%      2.111769e+06
max      2.898725e+06
Name: udp.stream, dtype: float64

#### 2.07.03 - `udp.time_delta`

- **Definition**: The time difference (offset) between the current frame and the previous frame in a specific UDP stream.
- **Treatment**: Retained.
- **Reason:** Sudden decreases in the time delta indicates a high-frequency packet injection (UDP Flood DDoS attacks).

In [1959]:
df[df['udp.stream'] > 0]['udp.stream'].describe()

count    1.453600e+04
mean     1.315078e+06
std      9.029611e+05
min      1.200000e+01
25%      4.968718e+05
50%      1.302127e+06
75%      2.111769e+06
max      2.898725e+06
Name: udp.stream, dtype: float64

### 2.08 - DNS (Domain Name System)

In [1960]:
cols = find_packet_type_columns(df, 'dns')
print("\n>>> DNS COLUMNS <<<\n", cols)


>>> DNS COLUMNS <<<
 ['dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in']


#### 2.08.01 - `dns.qry.name`, `dns.qry.name.len`, `dns.qry.qu` and, `dns.qry.type`

- **Definition**: The domain name being resolved into an IP address and its length. A boolean flag indicating a "QU" (unicast) question and a field specifying the type of DNS record requested.
- **Treatment**: Dropped.
- **Reason:** Misalignment during the feature extraction from raw PCAP files to CSV format.

In [1961]:
print_unique_values(df, ['dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type'])


--- dns.qry.name --- 
[0.0000000e+00 1.0000000e+00 3.2249918e+01 ... 2.8965490e+06 2.8968210e+06
 2.8969680e+06]


--- dns.qry.name.len --- 
<StringArray>
[                   '0.0',                    '1.0',                      '0',
  '0.debian.pool.ntp.org', '_googlecast._tcp.local',  '1.debian.pool.ntp.org',
  '2.debian.pool.ntp.org',  '3.debian.pool.ntp.org']
Length: 8, dtype: str


--- dns.qry.qu --- 
[   0.   37.   38.   70.   71.   95.   96.  120.  121.  166.  167.  197.
  198.  243.  244.  268.  271.  325.  326.  342.  346.  371.  372.  398.
  399.  420.  421.  441.  443.  476.  477.  517.  518.  574.  575.  599.
  600.  629.  630.  655.  656.  687.  688.  731.  732.  748.  751.  780.
  781.  801.  802.  828.  829.  848.  851.  908.  909.  925.  928.  954.
  957. 1001. 1027. 1028.   21.   22.]


--- dns.qry.type --- 
[0.]



In [1962]:
print(df[['dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type']].dtypes)

dns.qry.name        float64
dns.qry.name.len        str
dns.qry.qu          float64
dns.qry.type        float64
dtype: object


In [1963]:
df[~df['dns.qry.name.len'].isin(['0.0', '1.0', '0'])][['dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'Attack_label']].head()

,dns.qry.name,dns.qry.name.len,dns.qry.qu,dns.qry.type,Attack_label
124016,0.000000,0.debian.pool.ntp.org,21.0,0.0,0
126627,77217.553898,_googlecast._tcp.local,22.0,0.0,0
128839,0.000000,1.debian.pool.ntp.org,21.0,0.0,0
129150,0.000000,1.debian.pool.ntp.org,21.0,0.0,0
129151,0.002519,1.debian.pool.ntp.org,21.0,0.0,0


In [1964]:
for col in ['dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'dns.qry.name' column: True

Dropped 'dns.qry.name.len' column: True

Dropped 'dns.qry.qu' column: True

Dropped 'dns.qry.type' column: True


#### 2.08.02 - `dns.retransmission`

- **Definition**: Indicates if a DNS packet is a repeated transmission of a previous packet.
- **Treatment**: Dropped.
- **Reason:** Most entries are `0` so it provides almost no information.

In [1965]:
print_unique_values(df, ['dns.retransmission'])


--- dns.retransmission --- 
[ 0.  1. 12. 28.]



In [1966]:
df['dns.retransmission'].value_counts()

dns.retransmission
0.0     156957
1.0         21
28.0         7
12.0         1
Name: count, dtype: int64

In [1967]:
df[df['dns.retransmission'] > 0]['Attack_label'].value_counts()

Attack_label
0    29
Name: count, dtype: int64

In [1968]:
df.drop(columns=['dns.retransmission'], inplace=True)
print("\nDropped 'dns.retransmission' column:", not 'dns.retransmission' in df.columns) 


Dropped 'dns.retransmission' column: True


#### 2.08.03 - `dns.retransmit_request`

- **Definition**: A binary flag indicating whether the packet is specifically a retransmitted query (request)
- **Treatment**: Dropped.
- **Reason:** Only one entry is set to `1` so it provides no information.

In [1969]:
print_unique_values(df, ['dns.retransmit_request'])


--- dns.retransmit_request --- 
[0. 1.]



In [1970]:
df['dns.retransmit_request'].value_counts()

dns.retransmit_request
0.0    156985
1.0         1
Name: count, dtype: int64

In [1971]:
df[df['dns.retransmit_request'] == 1]['Attack_label']

128383    0
Name: Attack_label, dtype: int64

In [1972]:
df.drop(columns=['dns.retransmit_request'], inplace=True)
print("\nDropped 'dns.retransmit_request' column:", not 'dns.retransmit_request' in df.columns) 


Dropped 'dns.retransmit_request' column: True


#### 2.08.04 - `dns.retransmit_request_in`

- **Definition**: Contains the frame number of the original request that this packet is retransmitting.
- **Treatment**: Dropped.
- **Reason:** All values are set to `0` so it provides no information.

In [1973]:
print_unique_values(df, ['dns.retransmit_request_in'])


--- dns.retransmit_request_in --- 
[0.]



In [1974]:
df.drop(columns=['dns.retransmit_request_in'], inplace=True)
print("\nDropped 'dns.retransmit_request_in' column:", not 'dns.retransmit_request_in' in df.columns) 


Dropped 'dns.retransmit_request_in' column: True


### 2.09 - MQTT (Message Queuing Telemetry Transport)

In [1975]:
cols = find_packet_type_columns(df, 'mqtt')
print("\n>>> MQTT COLUMNS <<<\n", cols)


>>> MQTT COLUMNS <<<
 ['mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.protoname', 'mqtt.topic', 'mqtt.topic_len', 'mqtt.ver']


#### 2.09.01 - `mqtt.conack.flags`

- **Definition**: A hexadecimal or integer value representing the "Connect Acknowledgment" flags sent by the broker in response to a connection request.
- **Treatment**: Dropped.
- **Reason:** Constant to `0`. Provides no information.

In [1976]:
print_unique_values(df, ['mqtt.conack.flags'])


--- mqtt.conack.flags --- 
<StringArray>
['0.0', '0', '0x00000000']
Length: 3, dtype: str



In [1977]:
df.drop(columns=['mqtt.conack.flags'], inplace=True)
print("\nDropped 'mqtt.conack.flags' column:", not 'mqtt.conack.flags' in df.columns) 


Dropped 'mqtt.conack.flags' column: True


#### 2.09.02 - `mqtt.conflag.cleansess`

- **Definition**: A Boolean indicator that specifies whether the client and broker should discard previous session data and start a "clean" session.
- **Treatment**: Retained.
- **Reason:** Frequent "clean session" requests can be used in DoS attacks to force the broker to constantly reallocate resources.

In [1978]:
print_unique_values(df, ['mqtt.conflag.cleansess'])


--- mqtt.conflag.cleansess --- 
[0. 1.]



#### 2.09.03 - `mqtt.conflags`

- **Definition**: A composite field containing various flags from the MQTT CONNECT packet, such as Will Flag, Will QoS, and Will Retain.
- **Treatment**: Dropped.
- **Reason:** The only flag set is the Clean Session Flag which is already present in `mqtt.conflag.cleansess` colunm.

In [1979]:
print_unique_values(df, ['mqtt.conflags'])


--- mqtt.conflags --- 
[0. 2.]



In [1980]:
print("Correlation:", df[['mqtt.conflag.cleansess', 'mqtt.conflags']].corr()['mqtt.conflag.cleansess']['mqtt.conflags'])

Correlation: 1.0


In [1981]:
df.drop(columns=['mqtt.conflags'], inplace=True)
print("\nDropped 'mqtt.conflags' column:", not 'mqtt.conflags' in df.columns) 


Dropped 'mqtt.conflags' column: True


#### 2.09.04 - `mqtt.len`

- **Definition**: Numerical value representing the total length of the MQTT message payload.
- **Treatment**: Retained.
- **Reason:** Large or unexpected payload sizes are strong indicators of data exfiltration or buffer overflow attempts within Injection attacks.

In [1982]:
print_unique_values(df, ['mqtt.len'])


--- mqtt.len --- 
[ 0.  2. 12. 39.]



In [1983]:
df['mqtt.len'].describe()

count    156986.000000
mean          0.421515
std           3.615805
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          39.000000
Name: mqtt.len, dtype: float64

#### 2.09.05 - `mqtt.msg_decoded_as`

- **Definition**: A descriptive string identifying the format or structure into which the MQTT message payload was decoded.
- **Treatment**: Dropped.
- **Reason:** Constant to `0`. Provides no information.

In [1984]:
print_unique_values(df, ['mqtt.msg_decoded_as'])


--- mqtt.msg_decoded_as --- 
[0.]



In [1985]:
df.drop(columns=['mqtt.msg_decoded_as'], inplace=True)
print("\nDropped 'mqtt.msg_decoded_as' columns:", not 'mqtt.msg_decoded_as' in df.columns)


Dropped 'mqtt.msg_decoded_as' columns: True


#### 2.09.06 - `mqtt.msg`

- **Definition**: The raw content or sequence of bytes contained within the MQTT message payload
- **Treatment**: Retained / Replaced `'0.0'` and `'0'` to `''`
- **Reason:** Allows to detect Injection attacks carried inside the content of the MQTT payload.

In [1986]:
print_unique_values(df, ['mqtt.msg'])


--- mqtt.msg --- 
<StringArray>
[                       '0.0',                          '0',
 '32342e37392037362e36320d0a', '32342e36382037362e34320d0a',
 '32342e35372037362e32320d0a', '32342e35392037362e32370d0a',
 '32342e34382037362e30370d0a', '32342e32362037352e36370d0a',
 '32342e31352037352e34370d0a', '32342e31382037352e35320d0a',
 ...
 '32342e35322037362e31340d0a', '32332e34342037342e31390d0a',
 '32332e34372037342e32340d0a', '32332e35382037342e34340d0a',
 '32332e38352037342e39330d0a', '32332e38372037342e39380d0a',
 '32332e39302037352e30320d0a', '32332e39352037352e31320d0a',
 '32332e39382037352e31360d0a', '32342e30312037352e32310d0a']
Length: 117, dtype: str



In [1987]:
df['mqtt.msg'] = df['mqtt.msg'].replace(['0', '0.0'], '')

#### 2.09.07 - `mqtt.msgtype`

- **Definition**: An integer code identifying the type of MQTT control packet (e.g., 1 for CONNECT, 3 for PUBLISH).
- **Treatment**: Binarization.
- **Reason:** Critical for identifying protocol violations. A high frequency of CONNECT packets without subsequent PUBLISH actions may indicate a DoS attack.

In [1988]:
print_unique_values(df, ['mqtt.msgtype'])


--- mqtt.msgtype --- 
[ 0.  2.  1. 14.  3.]



In [1989]:
# MQTT v3.1.1 msg type codes
mqtt_types = {
    1: 'mqtt.msgtype_connect',
    2: 'mqtt.msgtype_connack',
    3: 'mqtt.msgtype_publish',
    4: 'mqtt.msgtype_puback',
    5: 'mqtt.msgtype_pubrec',
    6: 'mqtt.msgtype_pubrel',
    7: 'mqtt.msgtype_pubcomp',
    8: 'mqtt.msgtype_subscribe',
    9: 'mqtt.msgtype_suback',
    10: 'mqtt.msgtype_unsubscribe',
    11: 'mqtt.msgtype_unsuback',
    12: 'mqtt.msgtype_pingreq',
    13: 'mqtt.msgtype_pingresp',
    14: 'mqtt.msgtype_disconnect'
}

active_mqtt_columns = []
for code, col_name in mqtt_types.items():
    if (df['mqtt.msgtype'] == code).any():
        df[col_name] = (df['mqtt.msgtype'] == code).astype(int)
        active_mqtt_columns.append(col_name)

print(df[active_mqtt_columns].sum())

mqtt.msgtype_connect       1250
mqtt.msgtype_connack       1289
mqtt.msgtype_publish       1246
mqtt.msgtype_disconnect    1278
dtype: int64


#### 2.09.08 - `mqtt.hdrflags`

- **Definition**: An integer representation of the 8-bit fixed header byte. This byte is divided into two 4-bit sections: the Message Type (bits 7-4) and Specific Flags (bits 3-0) such as DUP (duplicate delivery), QoS (Quality of Service levels), and RETAIN.
- **Treatment**: Dropped.
- **Reason:** Only the msg type bits are set. Information already present in `mqtt.msgtype` column.

In [1990]:
print_unique_values(df, ['mqtt.hdrflags'])


--- mqtt.hdrflags --- 
[  0.  32.  16. 224.  48.]



In [1991]:
print([format(int(i), '08b') for i in df['mqtt.hdrflags'].unique()])

['00000000', '00100000', '00010000', '11100000', '00110000']


In [1992]:
print("mqtt.msgtype: ", [int(i) for i in df['mqtt.msgtype'].unique().astype(int)])
print("mqtt.hdrflags:", [int(i) >> 4 for i in df['mqtt.hdrflags'].unique()], "\t(4-bit shift to the right)")

mqtt.msgtype:  [0, 2, 1, 14, 3]
mqtt.hdrflags: [0, 2, 1, 14, 3] 	(4-bit shift to the right)


In [1993]:
df.drop(columns=['mqtt.hdrflags'], inplace=True)
print("\nDropped 'mqtt.hdrflags' columns:", not 'mqtt.hdrflags' in df.columns)


Dropped 'mqtt.hdrflags' columns: True


#### 2.09.09 - `mqtt.proto_len` and `mqtt.protoname`

- **Definition**: The length and string identifier of the protocol name
- **Treatment**: Dropped.
- **Reason:** Indicates if it is a MQTT packet or not. Provides no information.

In [1994]:
print_unique_values(df, ['mqtt.proto_len', 'mqtt.protoname'])


--- mqtt.proto_len --- 
[0. 4.]


--- mqtt.protoname --- 
<StringArray>
['0.0', '0', 'MQTT']
Length: 3, dtype: str



In [1995]:
df[['mqtt.proto_len', 'mqtt.protoname']].value_counts()

mqtt.proto_len  mqtt.protoname
0.0             0.0               132685
                0                  23051
4.0             MQTT                1250
Name: count, dtype: int64

In [1996]:
for col in ['mqtt.proto_len', 'mqtt.protoname']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'mqtt.proto_len' column: True

Dropped 'mqtt.protoname' column: True


#### 2.09.10 - `mqtt.topic` and `mqtt.topic_len`

- **Definition**: String representing the channel to which a message is published.
- **Treatment**: Dropped.
- **Reason:** Only one channel exists in this environment.

> NOTE
>
> In an environment with more than one channel, this will be a crucial feature to identify the purpose of the attack.

In [1997]:
print_unique_values(df, ['mqtt.topic', 'mqtt.topic_len'])


--- mqtt.topic --- 
<StringArray>
['0.0', '0', 'Temperature_and_Humidity']
Length: 3, dtype: str


--- mqtt.topic_len --- 
[ 0. 24.]



In [1998]:
for col in ['mqtt.topic', 'mqtt.topic_len']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'mqtt.topic' column: True

Dropped 'mqtt.topic_len' column: True


#### 2.09.11 - `mqtt.ver`

- **Definition**: The version number of the MQTT protocol being utilized.
- **Treatment**: Dropped.
- **Reason:** Every MQTT packet uses version 4 in this environment.

In [1999]:
print_unique_values(df, ['mqtt.ver'])


--- mqtt.ver --- 
[0. 4.]



In [2000]:
df.drop(columns=['mqtt.ver'], inplace=True)
print("\nDropped 'mqtt.ver' column:", not 'mqtt.ver' in df.columns)


Dropped 'mqtt.ver' column: True


### 2.10 - MBTCP (ModBus TCP)

In [2001]:
cols = find_packet_type_columns(df, 'mbtcp')
print("\n>>> MBTCP COLUMNS <<<\n", cols)


>>> MBTCP COLUMNS <<<
 ['mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']


#### 2.10.01 - `mbtcp.len`, `mbtcp.trans_id` and `mbtcp.unit_id`

- **Definition**: The number of bytes in the Modbus message, the numeric ID for client-server sync and the numeric ID of the remote slave.
- **Treatment**: Dropped
- **Reason:** All three features constant to `0`. Probably no ModBus packet is present in the dataset.

In [2002]:
print_unique_values(df, ['mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id'])


--- mbtcp.len --- 
[0.]


--- mbtcp.trans_id --- 
[0.]


--- mbtcp.unit_id --- 
[0.]



In [2003]:
for col in ['mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']:
    df.drop(columns=[col], inplace=True)
    print(f"\nDropped '{col}' column:", not col in df.columns)


Dropped 'mbtcp.len' column: True

Dropped 'mbtcp.trans_id' column: True

Dropped 'mbtcp.unit_id' column: True


## 3 - Columns after treatment

In [2004]:
print("Column count:", len(df.columns), "\t(Innitial count was 63)")
print("\n--- Columns ---\n", df.columns)

Column count: 40 	(Innitial count was 63)

--- Columns ---
 Index(['icmp.seq_le', 'http.file_data', 'http.content_length',
       'http.request.uri.query', 'http.referer', 'http.request.version',
       'http.response', 'http.tls_port', 'tcp.ack', 'tcp.len', 'tcp.seq',
       'udp.stream', 'udp.time_delta', 'mqtt.conflag.cleansess', 'mqtt.len',
       'mqtt.msg', 'mqtt.msgtype', 'Attack_label', 'Attack_type',
       'arp.opcode_request', 'arp.opcode_reply', 'http.request.method_get',
       'http.request.method_post', 'http.request.method_options',
       'http.request.method_trace', 'http.request.path', 'tcp.flag.res',
       'tcp.flag.ns', 'tcp.flag.cwr', 'tcp.flag.ece', 'tcp.flag.urg',
       'tcp.flag.ack', 'tcp.flag.psh', 'tcp.flag.rst', 'tcp.flag.syn',
       'tcp.flag.fin', 'mqtt.msgtype_connect', 'mqtt.msgtype_connack',
       'mqtt.msgtype_publish', 'mqtt.msgtype_disconnect'],
      dtype='str')


## 4 - Saving Data

In [2005]:
pd.to_pickle(df, config['processed_path'] / f"{filename}.pkl")